# Axes

```{note}
This is the **supporting experiments** notebook. It documents the exploration
that led to the conclusions on the [Storing axis metadata in xarray](axes.ipynb)
page. Start there for the summary and the two alternatives; come here for the
full reasoning and code behind each approach.
```

The `"axes"` field describes the dimensions of a physical coordinate space.
It is a list of dictionaries — one per array dimension — that names, classifies,
and optionally assigns physical units to each axis.

## Structure

Each axis dictionary has three fields:

| Field | Status | Type | Description |
|-------|--------|------|-------------|
| `name` | **MUST** | `str` | Unique name for this dimension |
| `type` | **SHOULD** | `str` | One of `"space"`, `"time"`, `"channel"`, or a custom string |
| `unit` | **SHOULD** | `str` | Physical unit (UDUNITS-2). Only meaningful for `"space"` and `"time"` |

`name` values must be unique across the axes list. They also define the Zarr v3
array's `dimension_names`.

## Axis types and ordering

The spec constrains both which types appear and their order within the list:

- **2 or 3** axes of type `"space"` (required)
- **0 or 1** axis of type `"time"` (optional)
- **0 or 1** axis of type `"channel"` or custom/null type (optional)
- Total: between 2 and 5 axes

The ordering rule: **time first** (if present), then **channel/custom** (if present),
then **space**. For 3 spatial axes the recommended order is `z, y, x`.

In [1]:
# A typical 5D axes list: TCZYX
axes_5d = [
    {"name": "t", "type": "time", "unit": "millisecond"},
    {"name": "c", "type": "channel"},
    {"name": "z", "type": "space", "unit": "micrometer"},
    {"name": "y", "type": "space", "unit": "micrometer"},
    {"name": "x", "type": "space", "unit": "micrometer"},
]
axes_5d

[{'name': 't', 'type': 'time', 'unit': 'millisecond'},
 {'name': 'c', 'type': 'channel'},
 {'name': 'z', 'type': 'space', 'unit': 'micrometer'},
 {'name': 'y', 'type': 'space', 'unit': 'micrometer'},
 {'name': 'x', 'type': 'space', 'unit': 'micrometer'}]

In [2]:
# Minimal 2D: just YX
axes_2d = [
    {"name": "y", "type": "space", "unit": "micrometer"},
    {"name": "x", "type": "space", "unit": "micrometer"},
]
axes_2d

[{'name': 'y', 'type': 'space', 'unit': 'micrometer'},
 {'name': 'x', 'type': 'space', 'unit': 'micrometer'}]

## Valid units

The spec restricts units to UDUNITS-2 strings. Channel axes have no defined units.

In [3]:
SPACE_UNITS = [
    "angstrom", "attometer", "centimeter", "decimeter", "exameter",
    "femtometer", "foot", "gigameter", "hectometer", "inch",
    "kilometer", "megameter", "meter", "micrometer", "mile",
    "millimeter", "nanometer", "parsec", "petameter", "picometer",
    "terameter", "yard", "yoctometer", "yottameter", "zeptometer",
    "zettameter",
]

TIME_UNITS = [
    "attosecond", "centisecond", "day", "decisecond", "exasecond",
    "femtosecond", "gigasecond", "hectosecond", "hour", "kilosecond",
    "megasecond", "microsecond", "millisecond", "minute", "nanosecond",
    "petasecond", "picosecond", "second", "terasecond", "yoctosecond",
    "yottasecond", "zeptosecond", "zettasecond",
]

print(f"{len(SPACE_UNITS)} space units, {len(TIME_UNITS)} time units")

26 space units, 23 time units


## Zarr v3 `dimension_names`

In NGFF 0.5, axis names must match the `dimension_names` field in the Zarr v3
array metadata. This is new in 0.5 — in 0.4 (Zarr v2), there was no
`dimension_names` field in the array metadata.

This redundancy is useful — `zarr-python` v3 can natively report dimension
names when opening an array, without parsing group-level `multiscales` metadata.

In [4]:
import zarr
from zarr.storage import MemoryStore

store = MemoryStore()
shape = (10, 3, 50, 256, 256)
dim_names = ["t", "c", "z", "y", "x"]

arr = zarr.open_array(
    store,
    mode="w",
    shape=shape,
    dtype="uint16",
    chunks=(1, 1, 10, 64, 64),
    dimension_names=dim_names,
)

# Read it back
arr2 = zarr.open_array(store, mode="r")
print(f"dimension_names from zarr: {arr2.metadata.dimension_names}")
assert list(arr2.metadata.dimension_names) == dim_names

dimension_names from zarr: ('t', 'c', 'z', 'y', 'x')


## Observations

- **`type` and `unit` are SHOULD, not MUST.** A reader must handle their absence.
- **Channel axes have no unit.** Channel is categorical, not continuous.
- **Custom axis types are allowed.** The spec MAY accepts any string for `type`.
- **Name choices are arbitrary.** Only uniqueness is required.

---

# Experiments

Now let's experiment with how to represent NGFF axes in xarray. The `name` → dim
mapping is obvious, but `type` and `unit` have no native home. We'll try
different approaches and see what works.

**Roadmap:**

1. **Dims only** — just use dimension names, discard type/unit
2. **Dataset attrs** — store type/unit dicts in `ds.attrs`
3. **Coordinate attrs** — store type/unit on each coordinate's `.attrs` (CF-style)
4. **Encoding dict** — use xarray's serialization encoding
5. **Raw JSON stash** — keep the original axes list verbatim in attrs
6. **Custom accessor** (v1 + v2) — typed API wrapping attrs storage
6b. **Scalar coordinates** — axis type as scalar (0-d) coordinates
8. **Round-trip constraints** — standard `to_zarr()` vs custom backend

In [5]:
import numpy as np
import xarray as xr

# Our running example: 5D TCZYX axes metadata + a shape
axes = [
    {"name": "t", "type": "time", "unit": "millisecond"},
    {"name": "c", "type": "channel"},
    {"name": "z", "type": "space", "unit": "micrometer"},
    {"name": "y", "type": "space", "unit": "micrometer"},
    {"name": "x", "type": "space", "unit": "micrometer"},
]
shape = (10, 3, 50, 256, 256)
dims = [a["name"] for a in axes]

## Experiment 1: dims only (no metadata)

What if we just map axis names to xarray dims and discard type/unit entirely?
This is the simplest possible mapping — let's see what we lose.

In [6]:
data = np.zeros(shape, dtype=np.uint16)
ds1 = xr.Dataset({"image": (dims, data)})
ds1

<xarray.Dataset> Size: 197MB
Dimensions:  (t: 10, c: 3, z: 50, y: 256, x: 256)
Dimensions without coordinates: t, c, z, y, x
Data variables:
    image    (t, c, z, y, x) uint16 197MB 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0

In [7]:
# This works — we get named dimensions
print("dims:", dict(ds1.sizes))

# But we've lost type and unit info. No way to know x is "space" in micrometers.
# Can we round-trip back to NGFF axes? Only the names.
recovered = [{"name": d} for d in ds1.sizes]
recovered

dims: {'t': 10, 'c': 3, 'z': 50, 'y': 256, 'x': 256}


[{'name': 't'}, {'name': 'c'}, {'name': 'z'}, {'name': 'y'}, {'name': 'x'}]

## Experiment 2: type and unit in Dataset attrs

What if we store axis metadata as dicts in `Dataset.attrs`? This keeps type/unit
together in one place, but Dataset attrs are fragile — let's test that.

In [8]:
ds2 = xr.Dataset({"image": (dims, data)})
ds2.attrs["axes_types"] = {a["name"]: a.get("type") for a in axes}
ds2.attrs["axes_units"] = {a["name"]: a["unit"] for a in axes if "unit" in a}
ds2

<xarray.Dataset> Size: 197MB
Dimensions:  (t: 10, c: 3, z: 50, y: 256, x: 256)
Dimensions without coordinates: t, c, z, y, x
Data variables:
    image    (t, c, z, y, x) uint16 197MB 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0
Attributes:
    axes_types:  {'t': 'time', 'c': 'channel', 'z': 'space', 'y': 'space', 'x...
    axes_units:  {'t': 'millisecond', 'z': 'micrometer', 'y': 'micrometer', '...

In [9]:
# Round-trip: reconstruct axes from the Dataset
def ds_to_axes(ds: xr.Dataset) -> list[dict]:
    types = ds.attrs.get("axes_types", {})
    units = ds.attrs.get("axes_units", {})
    result = []
    for dim_name in ds.sizes:
        ax = {"name": dim_name}
        if dim_name in types and types[dim_name] is not None:
            ax["type"] = types[dim_name]
        if dim_name in units:
            ax["unit"] = units[dim_name]
        result.append(ax)
    return result

recovered2 = ds_to_axes(ds2)
print("original: ", axes)
print("recovered:", recovered2)
assert recovered2 == axes

original:  [{'name': 't', 'type': 'time', 'unit': 'millisecond'}, {'name': 'c', 'type': 'channel'}, {'name': 'z', 'type': 'space', 'unit': 'micrometer'}, {'name': 'y', 'type': 'space', 'unit': 'micrometer'}, {'name': 'x', 'type': 'space', 'unit': 'micrometer'}]
recovered: [{'name': 't', 'type': 'time', 'unit': 'millisecond'}, {'name': 'c', 'type': 'channel'}, {'name': 'z', 'type': 'space', 'unit': 'micrometer'}, {'name': 'y', 'type': 'space', 'unit': 'micrometer'}, {'name': 'x', 'type': 'space', 'unit': 'micrometer'}]


Round-trips perfectly. But `attrs` are fragile — they don't survive many xarray
operations by default:

In [10]:
# Selection preserves Dataset attrs
sliced = ds2.sel(t=0)
print("after sel(t=0) attrs:", dict(sliced.attrs))

# But arithmetic drops them
added = ds2 + 1
print("after ds + 1 attrs:", dict(added.attrs))

after sel(t=0) attrs: {'axes_types': {'t': 'time', 'c': 'channel', 'z': 'space', 'y': 'space', 'x': 'space'}, 'axes_units': {'t': 'millisecond', 'z': 'micrometer', 'y': 'micrometer', 'x': 'micrometer'}}


after ds + 1 attrs: {'axes_types': {'t': 'time', 'c': 'channel', 'z': 'space', 'y': 'space', 'x': 'space'}, 'axes_units': {'t': 'millisecond', 'z': 'micrometer', 'y': 'micrometer', 'x': 'micrometer'}}


In [11]:
# xr.set_options(keep_attrs=True) can help, but it's global state
with xr.set_options(keep_attrs=True):
    added2 = ds2 + 1
    print("with keep_attrs:", dict(added2.attrs))

with keep_attrs: {'axes_types': {'t': 'time', 'c': 'channel', 'z': 'space', 'y': 'space', 'x': 'space'}, 'axes_units': {'t': 'millisecond', 'z': 'micrometer', 'y': 'micrometer', 'x': 'micrometer'}}


## Experiment 3: type and unit in coordinate attrs

What if we store axis metadata on each coordinate's `.attrs` dict — the same
pattern CF Conventions uses? This co-locates metadata with the coordinate it
describes.

In [12]:
ds3 = xr.Dataset({"image": (dims, data)})

# Assign dimension coordinates (just integer indices for now) with attrs
for ax in axes:
    name = ax["name"]
    size = shape[dims.index(name)]
    coord_attrs = {}
    if "type" in ax:
        coord_attrs["axis_type"] = ax["type"]
    if "unit" in ax:
        coord_attrs["units"] = ax["unit"]
    ds3 = ds3.assign_coords({name: (name, np.arange(size), coord_attrs)})

ds3

<xarray.Dataset> Size: 197MB
Dimensions:  (t: 10, c: 3, z: 50, y: 256, x: 256)
Coordinates:
  * t        (t) int64 80B 0 1 2 3 4 5 6 7 8 9
  * c        (c) int64 24B 0 1 2
  * z        (z) int64 400B 0 1 2 3 4 5 6 7 8 9 ... 41 42 43 44 45 46 47 48 49
  * y        (y) int64 2kB 0 1 2 3 4 5 6 7 8 ... 248 249 250 251 252 253 254 255
  * x        (x) int64 2kB 0 1 2 3 4 5 6 7 8 ... 248 249 250 251 252 253 254 255
Data variables:
    image    (t, c, z, y, x) uint16 197MB 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0

In [13]:
# Inspect individual coordinate attrs
for name in dims:
    print(f"{name}: {dict(ds3[name].attrs)}")

t: {'axis_type': 'time', 'units': 'millisecond'}
c: {'axis_type': 'channel'}
z: {'axis_type': 'space', 'units': 'micrometer'}
y: {'axis_type': 'space', 'units': 'micrometer'}
x: {'axis_type': 'space', 'units': 'micrometer'}


In [14]:
# Round-trip from coordinate attrs
def ds_to_axes_from_coords(ds: xr.Dataset) -> list[dict]:
    result = []
    for dim_name in ds.sizes:
        ax = {"name": dim_name}
        if dim_name in ds.coords:
            coord_attrs = ds[dim_name].attrs
            if "axis_type" in coord_attrs:
                ax["type"] = coord_attrs["axis_type"]
            if "units" in coord_attrs:
                ax["unit"] = coord_attrs["units"]
        result.append(ax)
    return result

recovered3 = ds_to_axes_from_coords(ds3)
print("recovered:", recovered3)
assert recovered3 == axes

recovered: [{'name': 't', 'type': 'time', 'unit': 'millisecond'}, {'name': 'c', 'type': 'channel'}, {'name': 'z', 'type': 'space', 'unit': 'micrometer'}, {'name': 'y', 'type': 'space', 'unit': 'micrometer'}, {'name': 'x', 'type': 'space', 'unit': 'micrometer'}]


This also round-trips. The advantage: metadata lives *on the coordinate itself*
rather than in a separate dict. It follows the CF conventions pattern where
`units` is a standard coordinate attribute.

Downside: requires that dimension coordinates exist. If we have bare dimensions
(no coordinates assigned), there's nowhere to put the attrs.

## Experiment 4: the encoding dict

What if we use xarray's `encoding` dict on Variables? It's designed for
serialization metadata — could it work for domain metadata too?

In [15]:
ds4 = xr.Dataset({"image": (dims, data)})

# encoding lives on Variables, not dims directly.
# We'd need coordinates to attach encoding to.
for ax in axes:
    name = ax["name"]
    size = shape[dims.index(name)]
    ds4 = ds4.assign_coords({name: np.arange(size)})

ds4["y"].encoding["axis_type"] = "space"
ds4["y"].encoding["units"] = "micrometer"
print("y encoding:", ds4["y"].encoding)

y encoding: {'axis_type': 'space', 'units': 'micrometer'}


In [16]:
# encoding is meant for serialization hints, not domain metadata.
# It doesn't survive most operations:
sliced4 = ds4.isel(y=slice(0, 128))
print("after isel, y encoding:", sliced4["y"].encoding)

after isel, y encoding: {'axis_type': 'space', 'units': 'micrometer'}


`encoding` survives some operations but it's not designed for this. It's meant
for round-trip serialization hints (dtype, compression). Using it for domain
metadata is a misuse of the API.

## Experiment 5: stash the raw axes list

What if we just keep the original NGFF JSON as-is in attrs? Perfect round-trip
by definition — but does it stay in sync with the Dataset?

In [17]:
ds5 = xr.Dataset({"image": (dims, data)})
ds5.attrs["ngff_axes"] = axes

# Perfect round-trip by definition
assert ds5.attrs["ngff_axes"] == axes
ds5.attrs["ngff_axes"]

[{'name': 't', 'type': 'time', 'unit': 'millisecond'},
 {'name': 'c', 'type': 'channel'},
 {'name': 'z', 'type': 'space', 'unit': 'micrometer'},
 {'name': 'y', 'type': 'space', 'unit': 'micrometer'},
 {'name': 'x', 'type': 'space', 'unit': 'micrometer'}]

In [18]:
# But it's opaque — xarray doesn't know what's in there.
# And it can get out of sync if you rename or drop dims:
renamed = ds5.rename({"x": "horizontal"})
print("dims:", list(renamed.sizes))
print("ngff_axes still says:", [a["name"] for a in renamed.attrs["ngff_axes"]])
# "x" in attrs, "horizontal" in dims — out of sync!

dims: ['t', 'c', 'z', 'y', 'horizontal']
ngff_axes still says: ['t', 'c', 'z', 'y', 'x']


## Experiment 6: a custom accessor

What if we build a structured, typed API using xarray's accessor mechanism?
The accessor reads from a well-known attr key but exposes `Axis` dataclass
instances — and can derive values from the Dataset's own dimensions.

In [19]:
from dataclasses import dataclass


@dataclass(frozen=True)
class Axis:
    """One NGFF axis."""
    name: str
    type: str | None = None
    unit: str | None = None

    def to_dict(self) -> dict:
        d = {"name": self.name}
        if self.type is not None:
            d["type"] = self.type
        if self.unit is not None:
            d["unit"] = self.unit
        return d


NGFF_AXES_ATTR = "_ngff_axes"


@xr.register_dataset_accessor("ngff")
class NGFFAccessor:
    """Accessor for NGFF metadata on an xarray Dataset."""

    def __init__(self, ds: xr.Dataset):
        self._ds = ds

    # --- axes metadata ---

    def _get_axes_raw(self) -> list[dict] | None:
        return self._ds.attrs.get(NGFF_AXES_ATTR)

    @property
    def axes(self) -> list[Axis] | None:
        """Return the NGFF axes, or None if not set."""
        raw = self._get_axes_raw()
        if raw is None:
            return None
        return [Axis(**a) for a in raw]

    @property
    def space_axes(self) -> list[Axis]:
        return [a for a in (self.axes or []) if a.type == "space"]

    @property
    def time_axes(self) -> list[Axis]:
        return [a for a in (self.axes or []) if a.type == "time"]

    @property
    def channel_axes(self) -> list[Axis]:
        return [a for a in (self.axes or []) if a.type == "channel"]

    def axis(self, name: str) -> Axis | None:
        """Look up one axis by name."""
        for a in (self.axes or []):
            if a.name == name:
                return a
        return None

    def to_axes_metadata(self) -> list[dict]:
        """Reconstruct the NGFF axes JSON from this Dataset."""
        if self.axes is not None:
            return [a.to_dict() for a in self.axes]
        # Fallback: just dim names
        return [{"name": d} for d in self._ds.sizes]


def make_ngff_dataset(data, dims, axes_meta, **kwargs) -> xr.Dataset:
    """Helper: build a Dataset and attach NGFF axes metadata."""
    ds = xr.Dataset({"image": (dims, data)}, **kwargs)
    ds.attrs[NGFF_AXES_ATTR] = axes_meta
    return ds

In [20]:
ds6 = make_ngff_dataset(data, dims, axes)
ds6

<xarray.Dataset> Size: 197MB
Dimensions:  (t: 10, c: 3, z: 50, y: 256, x: 256)
Dimensions without coordinates: t, c, z, y, x
Data variables:
    image    (t, c, z, y, x) uint16 197MB 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0
Attributes:
    _ngff_axes:  [{'name': 't', 'type': 'time', 'unit': 'millisecond'}, {'nam...

In [21]:
# Structured access to axes metadata
print("all axes:", ds6.ngff.axes)
print()
print("space axes:", ds6.ngff.space_axes)
print("time axes: ", ds6.ngff.time_axes)
print("channel:   ", ds6.ngff.channel_axes)

all axes: [Axis(name='t', type='time', unit='millisecond'), Axis(name='c', type='channel', unit=None), Axis(name='z', type='space', unit='micrometer'), Axis(name='y', type='space', unit='micrometer'), Axis(name='x', type='space', unit='micrometer')]

space axes: [Axis(name='z', type='space', unit='micrometer'), Axis(name='y', type='space', unit='micrometer'), Axis(name='x', type='space', unit='micrometer')]
time axes:  [Axis(name='t', type='time', unit='millisecond')]
channel:    [Axis(name='c', type='channel', unit=None)]


In [22]:
# Look up a single axis
y_axis = ds6.ngff.axis("y")
print(f"y axis: type={y_axis.type}, unit={y_axis.unit}")

c_axis = ds6.ngff.axis("c")
print(f"c axis: type={c_axis.type}, unit={c_axis.unit}")

y axis: type=space, unit=micrometer
c axis: type=channel, unit=None


In [23]:
# Round-trip back to NGFF JSON
recovered6 = ds6.ngff.to_axes_metadata()
print("recovered:", recovered6)
assert recovered6 == axes

recovered: [{'name': 't', 'type': 'time', 'unit': 'millisecond'}, {'name': 'c', 'type': 'channel'}, {'name': 'z', 'type': 'space', 'unit': 'micrometer'}, {'name': 'y', 'type': 'space', 'unit': 'micrometer'}, {'name': 'x', 'type': 'space', 'unit': 'micrometer'}]


Now let's test how it behaves under operations:

In [24]:
# sel preserves attrs → accessor still works
sliced6 = ds6.sel(t=0)
print("after sel(t=0):", sliced6.ngff.axes)

# isel too
isliced6 = ds6.isel(y=slice(0, 128))
print("after isel(y=...):", isliced6.ngff.axes)

after sel(t=0): [Axis(name='t', type='time', unit='millisecond'), Axis(name='c', type='channel', unit=None), Axis(name='z', type='space', unit='micrometer'), Axis(name='y', type='space', unit='micrometer'), Axis(name='x', type='space', unit='micrometer')]
after isel(y=...): [Axis(name='t', type='time', unit='millisecond'), Axis(name='c', type='channel', unit=None), Axis(name='z', type='space', unit='micrometer'), Axis(name='y', type='space', unit='micrometer'), Axis(name='x', type='space', unit='micrometer')]


In [25]:
# Arithmetic drops attrs by default — accessor returns None
added6 = ds6 + 1
print("after ds + 1:", added6.ngff.axes)

# With keep_attrs it survives
with xr.set_options(keep_attrs=True):
    added6_keep = ds6 + 1
print("with keep_attrs:", added6_keep.ngff.axes is not None)

after ds + 1: [Axis(name='t', type='time', unit='millisecond'), Axis(name='c', type='channel', unit=None), Axis(name='z', type='space', unit='micrometer'), Axis(name='y', type='space', unit='micrometer'), Axis(name='x', type='space', unit='micrometer')]
with keep_attrs: True


In [26]:
# rename — the attrs are still the old names
renamed6 = ds6.rename({"x": "horizontal"})
print("dims:", list(renamed6.sizes))
print("accessor axes:", [a.name for a in renamed6.ngff.axes])
# Still says 'x' — same sync problem as raw attrs

dims: ['t', 'c', 'z', 'y', 'horizontal']
accessor axes: ['t', 'c', 'z', 'y', 'x']


The accessor has the same underlying fragility as attrs (because it
*reads from* attrs). But it provides a better interface:

- **Typed access**: `Axis` dataclass instead of raw dicts
- **Filtering**: `.space_axes`, `.time_axes`, `.channel_axes`
- **Lookup**: `.axis("y")` returns `Axis(name='y', type='space', unit='micrometer')`
- **Round-trip**: `.to_axes_metadata()` produces the NGFF JSON
- **Extensible**: can grow to hold coordinateTransformations, omero, etc.

The rename sync issue could be solved by having the accessor
derive axis names from `ds.sizes` and only look up type/unit from
the stored metadata:

In [27]:
# A smarter accessor that derives names from dims, not stored metadata
@xr.register_dataset_accessor("ngff2")
class NGFFAccessorV2:
    def __init__(self, ds: xr.Dataset):
        self._ds = ds

    def _get_type_map(self) -> dict[str, str]:
        raw = self._ds.attrs.get(NGFF_AXES_ATTR, [])
        return {a["name"]: a.get("type") for a in raw}

    def _get_unit_map(self) -> dict[str, str]:
        raw = self._ds.attrs.get(NGFF_AXES_ATTR, [])
        return {a["name"]: a["unit"] for a in raw if "unit" in a}

    @property
    def axes(self) -> list[Axis]:
        """Build axes from current dims, looking up type/unit from stored metadata."""
        types = self._get_type_map()
        units = self._get_unit_map()
        return [
            Axis(name=d, type=types.get(d), unit=units.get(d))
            for d in self._ds.sizes
        ]

    def to_axes_metadata(self) -> list[dict]:
        return [a.to_dict() for a in self.axes]

In [28]:
# Now rename doesn't break — unknown dims just have no type/unit
renamed6v2 = ds6.rename({"x": "horizontal"})
print("v2 axes after rename:", renamed6v2.ngff2.axes)
print()
# 'horizontal' has no type/unit (not in the stored metadata),
# but the other 4 axes are correct
print("v2 round-trip:", renamed6v2.ngff2.to_axes_metadata())

v2 axes after rename: [Axis(name='t', type='time', unit='millisecond'), Axis(name='c', type='channel', unit=None), Axis(name='z', type='space', unit='micrometer'), Axis(name='y', type='space', unit='micrometer'), Axis(name='horizontal', type=None, unit=None)]

v2 round-trip: [{'name': 't', 'type': 'time', 'unit': 'millisecond'}, {'name': 'c', 'type': 'channel'}, {'name': 'z', 'type': 'space', 'unit': 'micrometer'}, {'name': 'y', 'type': 'space', 'unit': 'micrometer'}, {'name': 'horizontal'}]


## Experiment 6b: axis type as a scalar coordinate

What if axis type is just another coordinate? Axis type (`"space"`, `"time"`,
`"channel"`) is categorical data that maps 1:1 to dimensions. That's what a
scalar coordinate is — a label associated with a dimension. Just like channel
labels become a string coordinate on `c`, axis types can be scalar string
coordinates on each dimension.

In [29]:
ds6b = xr.Dataset({"image": (dims, data)})

# Assign axis types as scalar coordinates on each dimension
for ax in axes:
    name = ax["name"]
    size = shape[dims.index(name)]
    # The dimension coordinate (integer indices for now)
    ds6b = ds6b.assign_coords({name: np.arange(size)})
    # Axis type as a scalar coordinate associated with this dim
    if "type" in ax:
        ds6b = ds6b.assign_coords({f"{name}_type": ((), ax["type"])})

ds6b

<xarray.Dataset> Size: 197MB
Dimensions:  (t: 10, c: 3, z: 50, y: 256, x: 256)
Coordinates:
  * t        (t) int64 80B 0 1 2 3 4 5 6 7 8 9
  * c        (c) int64 24B 0 1 2
  * z        (z) int64 400B 0 1 2 3 4 5 6 7 8 9 ... 41 42 43 44 45 46 47 48 49
  * y        (y) int64 2kB 0 1 2 3 4 5 6 7 8 ... 248 249 250 251 252 253 254 255
  * x        (x) int64 2kB 0 1 2 3 4 5 6 7 8 ... 248 249 250 251 252 253 254 255
    t_type   <U4 16B 'time'
    c_type   <U7 28B 'channel'
    z_type   <U5 20B 'space'
    y_type   <U5 20B 'space'
    x_type   <U5 20B 'space'
Data variables:
    image    (t, c, z, y, x) uint16 197MB 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0

Hmm — scalar coordinates aren't associated with any dimension. They show
up under `Coordinates` but without a dim. Let's try a different approach:
make them 0-d coordinates that are explicitly tied to a dimension using
the dimension name.

In [30]:
# Alternative: use the dim name directly as the coordinate dimension
# so the type coordinate is associated with its axis
ds6b2 = xr.Dataset({"image": (dims, data)})

for ax in axes:
    name = ax["name"]
    size = shape[dims.index(name)]
    ds6b2 = ds6b2.assign_coords({name: np.arange(size)})

# Axis types as a 1-element coordinate on a new "axis" dimension?
# No — that doesn't make sense. Let's try coordinate attrs instead
# for type, but use a real coordinate for unit since unit is more
# like data (you might want to do unit conversion).

# Actually, the simplest version: just put type in coord attrs,
# and also try unit as coord attrs
for ax in axes:
    name = ax["name"]
    if "type" in ax:
        ds6b2[name].attrs["axis_type"] = ax["type"]
    if "unit" in ax:
        ds6b2[name].attrs["units"] = ax["unit"]

ds6b2

<xarray.Dataset> Size: 197MB
Dimensions:  (t: 10, c: 3, z: 50, y: 256, x: 256)
Coordinates:
  * t        (t) int64 80B 0 1 2 3 4 5 6 7 8 9
  * c        (c) int64 24B 0 1 2
  * z        (z) int64 400B 0 1 2 3 4 5 6 7 8 9 ... 41 42 43 44 45 46 47 48 49
  * y        (y) int64 2kB 0 1 2 3 4 5 6 7 8 ... 248 249 250 251 252 253 254 255
  * x        (x) int64 2kB 0 1 2 3 4 5 6 7 8 ... 248 249 250 251 252 253 254 255
Data variables:
    image    (t, c, z, y, x) uint16 197MB 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0

In [31]:
# What about using a 1-element array coordinate on the dimension?
# This ties the type to the dimension explicitly.
ds6b3 = xr.Dataset({"image": (dims, data)})

for ax in axes:
    name = ax["name"]
    size = shape[dims.index(name)]
    ds6b3 = ds6b3.assign_coords({name: np.arange(size)})
    if "type" in ax:
        # A single-element coordinate on this dim
        # This broadcasts: every element of the dim has the same type
        ds6b3.coords[f"{name}_axis_type"] = (name, np.full(size, ax["type"]))

ds6b3

<xarray.Dataset> Size: 197MB
Dimensions:      (t: 10, c: 3, z: 50, y: 256, x: 256)
Coordinates:
  * t            (t) int64 80B 0 1 2 3 4 5 6 7 8 9
    t_axis_type  (t) <U4 160B 'time' 'time' 'time' ... 'time' 'time' 'time'
  * c            (c) int64 24B 0 1 2
    c_axis_type  (c) <U7 84B 'channel' 'channel' 'channel'
  * z            (z) int64 400B 0 1 2 3 4 5 6 7 8 ... 41 42 43 44 45 46 47 48 49
    z_axis_type  (z) <U5 1kB 'space' 'space' 'space' ... 'space' 'space' 'space'
  * y            (y) int64 2kB 0 1 2 3 4 5 6 7 ... 249 250 251 252 253 254 255
    y_axis_type  (y) <U5 5kB 'space' 'space' 'space' ... 'space' 'space' 'space'
  * x            (x) int64 2kB 0 1 2 3 4 5 6 7 ... 249 250 251 252 253 254 255
    x_axis_type  (x) <U5 5kB 'space' 'space' 'space' ... 'space' 'space' 'space'
Data variables:
    image        (t, c, z, y, x) uint16 197MB 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0

That works but wastes memory repeating `"space"` 256 times for x and y.
And it conflates axis metadata with data coordinates.

What if we use a scalar (0-d) coordinate? xarray supports 0-d coordinates
that are associated with no dimension:

In [32]:
ds6b4 = xr.Dataset({"image": (dims, data)})

for ax in axes:
    name = ax["name"]
    size = shape[dims.index(name)]
    ds6b4 = ds6b4.assign_coords({name: np.arange(size)})

# Axis types as scalar coordinates
axis_type_map = {ax["name"]: ax["type"] for ax in axes if "type" in ax}
axis_unit_map = {ax["name"]: ax["unit"] for ax in axes if "unit" in ax}

# One scalar coord per axis, named descriptively
for dim_name, axis_type in axis_type_map.items():
    ds6b4.coords[f"{dim_name}_axis_type"] = axis_type
for dim_name, unit in axis_unit_map.items():
    ds6b4.coords[f"{dim_name}_axis_unit"] = unit

ds6b4

<xarray.Dataset> Size: 197MB
Dimensions:      (t: 10, c: 3, z: 50, y: 256, x: 256)
Coordinates: (12/14)
  * t            (t) int64 80B 0 1 2 3 4 5 6 7 8 9
  * c            (c) int64 24B 0 1 2
  * z            (z) int64 400B 0 1 2 3 4 5 6 7 8 ... 41 42 43 44 45 46 47 48 49
  * y            (y) int64 2kB 0 1 2 3 4 5 6 7 ... 249 250 251 252 253 254 255
  * x            (x) int64 2kB 0 1 2 3 4 5 6 7 ... 249 250 251 252 253 254 255
    t_axis_type  <U4 16B 'time'
    ...           ...
    y_axis_type  <U5 20B 'space'
    x_axis_type  <U5 20B 'space'
    t_axis_unit  <U11 44B 'millisecond'
    z_axis_unit  <U10 40B 'micrometer'
    y_axis_unit  <U10 40B 'micrometer'
    x_axis_unit  <U10 40B 'micrometer'
Data variables:
    image        (t, c, z, y, x) uint16 197MB 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0

In [33]:
# These are real coordinates — they survive operations
sliced = ds6b4.isel(y=slice(0, 128))
print("after isel:", sliced.coords["y_axis_type"].values)
print("after isel:", sliced.coords["y_axis_unit"].values)

# And they're accessible
print()
for d in dims:
    type_key = f"{d}_axis_type"
    unit_key = f"{d}_axis_unit"
    t = str(ds6b4.coords[type_key].values) if type_key in ds6b4.coords else None
    u = str(ds6b4.coords[unit_key].values) if unit_key in ds6b4.coords else None
    print(f"{d}: type={t}, unit={u}")

after isel: space
after isel: micrometer

t: type=time, unit=millisecond
c: type=channel, unit=None
z: type=space, unit=micrometer
y: type=space, unit=micrometer
x: type=space, unit=micrometer


In [34]:
# Do they survive to_zarr and back?
store_6b = MemoryStore()
ds6b4.to_zarr(store_6b, mode="w")
ds6b4_rt = xr.open_zarr(store_6b)

print("round-tripped:")
for d in dims:
    type_key = f"{d}_axis_type"
    unit_key = f"{d}_axis_unit"
    t = str(ds6b4_rt.coords[type_key].values) if type_key in ds6b4_rt.coords else None
    u = str(ds6b4_rt.coords[unit_key].values) if unit_key in ds6b4_rt.coords else None
    print(f"  {d}: type={t}, unit={u}")

/Users/ian/Documents/dev/xarray-ngff/research/.venv/lib/python3.12/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=4, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)
/Users/ian/Documents/dev/xarray-ngff/research/.venv/lib/python3.12/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=5, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change with

round-tripped:
  t: type=time, unit=millisecond
  c: type=channel, unit=None
  z: type=space, unit=micrometer
  y: type=space, unit=micrometer
  x: type=space, unit=micrometer


/Users/ian/Documents/dev/xarray-ngff/research/.venv/lib/python3.12/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=10, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)
/Users/ian/Documents/dev/xarray-ngff/research/.venv/lib/python3.12/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=11, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change wi

Scalar coordinates survive `to_zarr()` round-trips. They're real data, not
metadata — xarray treats them as first-class citizens.

**Pros:**
- Survives operations (sel, isel, arithmetic with `keep_attrs`)
- Survives `to_zarr()` / `open_zarr()` round-trips
- No attrs needed — it's all coordinates
- Discoverable: shows up in the Dataset repr

**Cons:**
- Naming convention needed (`{dim}_axis_type`, `{dim}_axis_unit`)
- Clutters the coordinate namespace (10 extra coords for a 5D dataset)
- Scalar coords aren't *associated with* a dimension — the association
  is only by naming convention
- Not standard NGFF — a custom backend would still need to know
  how to interpret these

## Experiment 7: (skipped)

*There is no Experiment 7 — the numbering jumps from 6b to 8. The "6b" label
was added when scalar coordinates emerged as a variant of Experiment 6 (accessor),
and the next natural topic was round-trip constraints.*

## Experiment 8: round-trip constraints

The most important question: **are we constrained to standard `ds.to_zarr()` /
`xr.open_zarr()`, or can we use a custom backend?** Where we store metadata
depends entirely on the answer. Let's explore both paths.

### Path A: standard xarray-to-zarr

With `ds.to_zarr()`, xarray writes:
- Array data → zarr arrays
- Dimension coordinates → zarr arrays with matching names
- `.attrs` → zarr group/array attributes (must be JSON-serializable)
- `.encoding` → influences zarr codec/chunk config

That's it. There's no hook to write NGFF-structured metadata into
the `ome` namespace in `zarr.json`. So our only options for axis
type/unit are attrs or coordinate attrs — and we'd need a separate
step to produce valid NGFF metadata.

Let's try it:

In [35]:
import zarr
from zarr.storage import MemoryStore

# Build a dataset with axis metadata in attrs
ds_a = make_ngff_dataset(data, dims, axes)

# Write with standard xarray
store_a = MemoryStore()
ds_a.to_zarr(store_a, mode="w")

# What got written?
root = zarr.open_group(store_a, mode="r")
print("group attrs:", dict(root.attrs))
print()
print("arrays:", list(root.array_keys()))
print("groups:", list(root.group_keys()))

group attrs: {'_ngff_axes': [{'name': 't', 'type': 'time', 'unit': 'millisecond'}, {'name': 'c', 'type': 'channel'}, {'name': 'z', 'type': 'space', 'unit': 'micrometer'}, {'name': 'y', 'type': 'space', 'unit': 'micrometer'}, {'name': 'x', 'type': 'space', 'unit': 'micrometer'}]}

arrays: ['image']
groups: []


/Users/ian/Documents/dev/xarray-ngff/research/.venv/lib/python3.12/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


In [36]:
# Read it back
ds_a_rt = xr.open_zarr(store_a)
print("round-tripped attrs:", dict(ds_a_rt.attrs))
print("ngff axes:", ds_a_rt.ngff.axes)

round-tripped attrs: {'_ngff_axes': [{'name': 't', 'type': 'time', 'unit': 'millisecond'}, {'name': 'c', 'type': 'channel'}, {'name': 'z', 'type': 'space', 'unit': 'micrometer'}, {'name': 'y', 'type': 'space', 'unit': 'micrometer'}, {'name': 'x', 'type': 'space', 'unit': 'micrometer'}]}
ngff axes: [Axis(name='t', type='time', unit='millisecond'), Axis(name='c', type='channel', unit=None), Axis(name='z', type='space', unit='micrometer'), Axis(name='y', type='space', unit='micrometer'), Axis(name='x', type='space', unit='micrometer')]


The attrs round-trip through standard zarr. But the zarr store has
*no* NGFF structure — no `ome` namespace, no `multiscales`, no
`coordinateTransformations`. It's just a flat zarr store with xarray
conventions. An NGFF reader wouldn't recognize it.

To produce valid NGFF we'd need to:
1. Write with `to_zarr()`
2. Post-process: rewrite `zarr.json` to inject `ome.multiscales` metadata
3. Hope nothing gets out of sync

This is fragile.

### Path B: custom backend

With a custom xarray backend engine (`engine="ome-zarr"`), we control
both read and write. The backend can:
- **Read**: parse `ome.multiscales` → populate dims, coords, attrs, indexes
- **Write**: take dims, coords, attrs, indexes → produce valid `ome.multiscales`

This means the metadata can live *anywhere* in xarray's data model
because the backend knows how to reconstruct the NGFF JSON from
whatever representation we choose.

The key insight: **with a custom backend, we don't need attrs to
survive serialization — the backend IS the serialization layer.**

In [37]:
# Sketch of what a custom backend write path looks like
def write_ngff(ds: xr.Dataset, store) -> None:
    """Write an xarray Dataset as an NGFF 0.5 store."""
    # Reconstruct axes metadata from the Dataset
    # Option 1: from accessor (if available)
    if ds.ngff.axes is not None:
        axes_meta = ds.ngff.to_axes_metadata()
    else:
        # Option 2: from coordinate attrs (CF-style)
        axes_meta = []
        for dim_name in ds.sizes:
            ax = {"name": dim_name}
            if dim_name in ds.coords:
                attrs = ds[dim_name].attrs
                if "axis_type" in attrs:
                    ax["type"] = attrs["axis_type"]
                if "units" in attrs:
                    ax["unit"] = attrs["units"]
            axes_meta.append(ax)

    print("would write axes:", axes_meta)
    # ... then write zarr arrays + inject ome.multiscales into zarr.json

# Works with either approach
write_ngff(ds6, None)  # from accessor
write_ngff(ds3, None)  # from coordinate attrs

would write axes: [{'name': 't', 'type': 'time', 'unit': 'millisecond'}, {'name': 'c', 'type': 'channel'}, {'name': 'z', 'type': 'space', 'unit': 'micrometer'}, {'name': 'y', 'type': 'space', 'unit': 'micrometer'}, {'name': 'x', 'type': 'space', 'unit': 'micrometer'}]
would write axes: [{'name': 't', 'type': 'time', 'unit': 'millisecond'}, {'name': 'c', 'type': 'channel'}, {'name': 'z', 'type': 'space', 'unit': 'micrometer'}, {'name': 'y', 'type': 'space', 'unit': 'micrometer'}, {'name': 'x', 'type': 'space', 'unit': 'micrometer'}]


### What this means for where we store axis metadata

| Constraint | Where to store type/unit | Round-trip mechanism |
|------------|------------------------|---------------------|
| Standard `to_zarr` only | Must be in attrs (only thing that persists) | attrs → zarr attrs → attrs |
| Custom backend | **Anywhere** — dims, coords, attrs, accessor, index | Backend reads/writes NGFF directly |

With a custom backend:
- Axis `name` lives in dims (native)
- Axis `type` and `unit` can live in coordinate attrs (CF-style)
  or an accessor or even just be reconstructed from conventions
- The backend handles the NGFF ↔ xarray translation on read/write
- No need to stash raw JSON for round-trip fidelity

**This is the approach we should design for.** The accessor becomes
a convenience API for *in-memory* access, not a persistence mechanism.
The backend handles persistence.

## Summary

### Design requirements

Whatever representation we choose must satisfy three constraints:

1. **Robust to user mistakes** — raw dicts in attrs are easy to
   corrupt, typo, or forget. The API should make wrong things hard.
2. **Usable for developers** — structured, typed, discoverable.
   Developers should be able to inspect and manipulate axis metadata
   without reading docs about which attrs key to look in.
3. **Never loses data** — every field in the NGFF axes spec must
   survive the full read → manipulate → write cycle, even for fields
   we don't understand (forward compatibility).

### What we learned

| Approach | Robust? | Developer UX | Never loses data? |
|----------|---------|-------------|-------------------|
| Dims only | n/a | good (native) | no (loses type/unit) |
| Dataset attrs | no (raw dicts) | poor | yes (if stashed) |
| Coordinate attrs | better | ok (CF-like) | partial (must exist) |
| Raw JSON stash | no (opaque blob) | poor | yes |
| Accessor over attrs | better (typed) | good | depends on backing |

### Proposed approach

**Axis names → dims** is unambiguous and native. This is the easy part.

For **type and unit**, the best option depends on the write path:

- **Custom backend** (recommended): The backend reconstructs NGFF JSON
  from the xarray representation on write. Axis metadata can live in
  coordinate attrs (CF-style) or a typed accessor — the backend knows
  how to find it. This decouples in-memory representation from on-disk
  format.

- **Standard `to_zarr()`**: We're constrained to attrs. Must stash
  enough information to reconstruct NGFF, which means raw JSON or
  structured attrs. Either way, a post-processing step is needed
  to produce valid NGFF.

A **typed accessor** (like the v2 pattern above) provides the developer
UX layer regardless of storage. It should:
- Derive names from dims (always in sync)
- Provide typed `Axis` objects, not raw dicts
- Validate on construction (catch typos in type/unit early)
- Preserve unknown fields for forward compatibility

### Open questions for later pages

- How do `coordinateTransformations` interact with this? (next page)
- Should the accessor live on Dataset, DataTree, or both?
- Can a custom `Index` carry axis metadata instead of attrs?
- What's the right boundary between accessor and backend?

---

For the conclusions drawn from these experiments — including the two alternatives
presented for community input — see [Storing axis metadata in xarray](axes.ipynb).